# 4. Real time

In notebook 3, the chunks came from files. Here they come live, packet by packet, as an
interrogator sends them. There is no interrogator behind this notebook: `replay.py` plays the
two cables as if they were live, in an endless loop of 3.5 min, stamped with the current UTC
time. The earthquake comes back at every loop.

Before you start, open a terminal in this folder and leave this running (`--waterfall` also
shows the last minute of both cables, in a window of its own):

```
uv run replay.py --waterfall
```

| Stream | Address | Array |
|---|---|---|
| N | `tcp://localhost:5556` | `(time, distance)`, 100 Hz, packets of 24 samples (0.24 s) |
| S | `tcp://localhost:5557` | `(time, distance)`, 62.5 Hz, packets of 24 samples (0.384 s) |

Like a real interrogator, the replay sends packets of a fixed number of samples: their
duration depends on the sampling rate of the cable.

In this notebook you will:
1. receive packets from a stream,
2. pick them one at a time with the pipeline of notebook 3,
3. follow both cables live, with a dashboard that updates itself,
4. measure how fast the earthquake is located.

> `uv run replay.py --speedup 4` plays four times faster: handy to test, but the stream times
> then run ahead of the clock. In the terminal of the replay, press `e` to jump to just before
> the earthquake, instead of waiting for it to come back.

In [1]:
# solutions only: run from the exercises folder
%cd ..

/Users/trabatto/pkgs/exercises


In [2]:
import time

import numpy as np
import pandas as pd
import scipy.signal as sp
import xdas as xd
from xdas.atoms import Atom, Partial, State
from xdas.io.asn import ZMQSubscriber

from tools import EEWS, Dashboard, PeakAmplitude, poll

What you wrote in notebook 3, as a single atom: `make_picker` chains everything, from the
raw packets to the picks with their P amplitude. It is a function because each cable needs its
own picker: each atom keeps the state of one stream.

The trigger only keeps the distance and the time of the picks: the packets have no geometry
(see below).

In [3]:
gain = 1536.6e-9 / (4 * np.pi * 1.4677 * 0.78)  # from notebook 2


def remove_common_mode(da):
    return da - da.median("distance")


class STALTA(Atom):
    def __init__(self, sta, lta):
        super().__init__()
        self.sta = sta
        self.lta = lta
        self.short = State(...)
        self.long = State(...)
        self.count = State(...)

    def initialize(self, da, **flags):
        self.short = State(np.zeros((1, da.sizes["distance"])))
        self.long = State(np.zeros((1, da.sizes["distance"])))
        self.count = State(0)

    def call(self, da, **flags):
        dt = da["time"].get_sampling_interval()
        csta, clta = dt / self.sta, dt / self.lta
        power = da.values ** 2
        short, zshort = sp.lfilter([csta], [1, csta - 1], power, axis=0, zi=self.short)
        long, zlong = sp.lfilter([clta], [1, clta - 1], power, axis=0, zi=self.long)
        self.short = State(zshort)
        self.long = State(zlong)
        ratio = short / long
        ratio[: max(round(self.lta / dt) - self.count, 0)] = 0  # the long average is not settled yet
        self.count = State(self.count + len(ratio))
        return da.copy(data=ratio)


def make_picker():
    return (
        Partial(np.multiply, gain)
        >> Partial(remove_common_mode)
        >> xd.filter(..., (2.0, 15.0), dim="time")
        >> xd.resample(..., interval=45.0, snap=True, dim="distance")
        >> xd.integrate(..., dim="time")
        >> xd.integrate(..., dim="distance")
        >> xd.sliding_mean_removal(..., wlen=2000.0, dim="distance")
        >> PeakAmplitude(
            STALTA(sta=0.2, lta=5.0) >> xd.trigger(..., thresh=8.0, coords=["distance", "time"]),
            window=2.0,
        )
    )


abyss = xd.open("abyss.nc")
keep = {"N": slice(5000, 97000), "S": slice(5000, 130000)}
ports = {"N": 5556, "S": 5557}

## Listen

The replay speaks the protocol of an ASN OptoDAS interrogator, as a real one would. A
`ZMQSubscriber` connects to a stream; it is an iterator: each `next` waits for the next packet
and returns it as a `DataArray`.

**Task:** subscribe to the north cable and receive one packet. What are its dimensions, its
duration, its coordinates? How late is its last sample, compared with the clock?

> **Reminder:**
> ```python
> subscriber = ZMQSubscriber("tcp://localhost:1234")   # connect to the stream on port 1234
> packet = next(subscriber)                            # wait for the next packet
> np.datetime64("now", "ns")                           # the clock (UTC)
> da["time"].values[-1]                                # the last time of da
> ```

📖 [Streaming](https://xdas.readthedocs.io/en/stable/user-guide/pipeline/streaming.html) ·
[`xdas.io.asn.ZMQSubscriber`](https://xdas.readthedocs.io/en/stable/_autosummary/xdas.io.asn.ZMQSubscriber.html)

In [4]:
subscriber = ZMQSubscriber(f"tcp://localhost:{ports['N']}")
packet = next(subscriber)
print(np.datetime64("now", "ns") - packet["time"].values[-1])
subscriber.close()  # or the packets pile up in memory
packet

-230000000 nanoseconds


<xdas.DataArray (time: 24, distance: 6664)>
[[ 1.728816e+01  1.744016e+01  1.630631e-01 ... -1.542663e+00
  -3.691314e-01  1.644149e+00]
 [ 2.411374e+01  2.402062e+01 -9.606531e-02 ... -1.526363e+00
  -1.241517e+00 -2.061665e+00]
 [-2.750690e+01 -2.756071e+01 -4.977712e-02 ... -1.905648e+00
  -2.408703e+00 -1.007654e+00]
 ...
 [-1.284195e+01 -1.282391e+01  2.033382e-02 ...  7.489860e-02
  -2.141504e+00 -3.784928e+00]
 [ 1.144189e+00  1.117310e+00 -2.623535e-02 ...  4.951259e-01
  -4.690816e-02 -3.025633e-01]
 [ 1.981071e+01  1.982003e+01  6.651082e-03 ...  2.830443e-01
   1.801094e+00  1.608471e+00]]
Coordinates:
  * time (time): 2026-10-25T16:10:59.000 to 2026-10-25T16:10:59.230
  * distance (distance): 0.000 to 102073.848

## Packet by packet

The interrogator only knows time and distance along the fiber: where each channel lies is our
metadata, attached in notebook 1. But only the picks need a position, to locate the
earthquake, and they are few: the early-warning system will look them up in `abyss`. The
packets only need their channels in the sea (`keep`).

A pipeline called with `chunk_dim="time"` takes the next chunk of a stream and keeps its state:
exactly what we need for packets.

**Task:** for 10 packets of the north cable, keep the channels in the sea, run the picker on
each packet, and print the number of picks and the processing time. Is it faster than the
0.24 s between two packets?

> **Reminder:**
> ```python
> for packet in subscriber:          # loop over the packets as they come (never ends)
>     ...
>     if count == 10:
>         break                      # leave the loop
> tic = time.time()                  # the clock in seconds; time.time() - tic: elapsed time
> ```

In [5]:
subscriber = ZMQSubscriber(f"tcp://localhost:{ports['N']}")
picker = make_picker()
for count, packet in enumerate(subscriber, start=1):
    tic = time.time()
    picks = picker(packet.sel(distance=keep["N"]), chunk_dim="time")
    print(packet["time"].values[0], len(picks), "picks", f"{time.time() - tic:.3f} s")
    if count == 10:
        break
subscriber.close()

2026-10-25T16:10:59.240000000 0 picks 0.023 s


2026-10-25T16:10:59.480000000 0 picks 0.018 s


2026-10-25T16:10:59.720000000 0 picks 0.020 s


2026-10-25T16:10:59.960000000 0 picks 0.019 s


2026-10-25T16:11:00.200000000 0 picks 0.017 s


...

2026-10-25T16:11:10.280000000 0 picks 0.020 s


2026-10-25T16:11:10.520000000 2 picks 0.020 s


2026-10-25T16:11:10.760000000 0 picks 0.020 s


2026-10-25T16:11:11.000000000 0 picks 0.020 s


2026-10-25T16:11:11.240000000 0 picks 0.020 s


## Live

Now both cables at once. The two streams do not tick together (a packet every 0.24 s on N,
every 0.384 s on S): `poll` (in `tools.py`) listens to both, and gives each packet as soon as it
arrives, with the name of its cable.

The display is given: `Dashboard` (in `tools.py`) shows the picks of the last minute on each
cable, the solution on a map against EMSC (gold star), and how the solution evolves: its
distance to EMSC and its magnitude. It is a plotly figure that redraws itself in place, at most
twice a second, so it does not slow the loop down. To see the waves too, start the replay with
`uv run replay.py --waterfall`: it shows the last minute of both cables in a window of its own.

In [ ]:
emsc = {"longitude": -71.800, "latitude": -29.954, "depth": 50000.0, "magnitude": 3.4}
dashboard = Dashboard(abyss, emsc, topo=xd.open("data/topo/GMRT.nc"))
dashboard

**Task:** complete the loop. For each packet: keep the channels in the sea, pick with the
picker of its cable, add the cable to the picks, and give them to the early-warning system
(which looks up their positions in `abyss`). Keep its solutions, and update the dashboard. The
loop runs forever: watch the picks sweep the cables, then the solution settle, and stop it with
the ■ button (interrupt the kernel) once you have seen the earthquake. It comes back every
3.5 min (or press `e` in the replay terminal).

> **Reminder:**
> ```python
> for cable, packet in poll(subscribers):    # the packets of all streams as they come (never ends)
> table["column"] = value                    # a column with the same value on every row
> rows = eews(picks, now)                    # the new solutions, maybe none (notebook 3)
> dashboard.update(picks, rows, now)         # show them
> ```

In [7]:
subscribers = {cable: ZMQSubscriber(f"tcp://localhost:{port}") for cable, port in ports.items()}
pickers = {cable: make_picker() for cable in subscribers}
eews = EEWS(geometry=abyss)
solutions = []

try:
    for cable, packet in poll(subscribers):
        now = packet["time"].values[-1]
        picks = pickers[cable](packet.sel(distance=keep[cable]), chunk_dim="time")
        picks["cable"] = cable
        rows = eews(picks, now)
        solutions.append(rows)
        dashboard.update(picks, rows, now)
except KeyboardInterrupt:
    pass  # stopped with the ■ button: keep what was found
finally:
    for subscriber in subscribers.values():
        subscriber.close()
solutions = pd.concat(solutions, ignore_index=True)

## How fast?

`solutions` holds all the solutions of the loop, as in notebook 3, with the stream time at which
each came out (`time`) and the origin time it found (`origin`).

**Task:** for the first earthquake of the run, how many seconds after the origin time (the one
of its last solution, the best) did its first solution come? Where does that
time go (the travel time of the P wave, the picking, the 2 s window of `PeakAmplitude`, ...)?
Try `EEWS(min_picks=...)` lower and higher, and run the loop again. What do you gain, and what do
you lose?

In [8]:
located = solutions.dropna()
first = located.iloc[0]
event = located[located["time"] < first["time"] + np.timedelta64(30, "s")]  # the first earthquake only
last = event.iloc[-1]
print(first)
(first["time"] - last["origin"]).total_seconds()

time         2026-10-25 16:11:56.110000
origin       2026-10-25 16:11:39.760000
longitude                        -72.05
latitude                         -29.85
depth                           85000.0
magnitude                      3.780473
picks                              1006
Name: 44, dtype: object


10.208